Part 1 : Data Import

In [30]:
import pandas as pd
import numpy as np
from scipy import stats


# Load the dataset into a pandas DataFrame
try:
    data = pd.read_csv('data.csv', sep='\t')
    print("Dataset has been successfully loaded.")
except Exception as e:
    print(f"Failed to load data: {e}")


# Display the first few rows of the dataset
print(data.head())
# Separate numeric columns from non-numeric columns
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
non_numeric_cols = data.select_dtypes(exclude=['int64', 'float64']).columns

/tmp/ipykernel_3546/1550843614.py:8: DtypeWarning: Columns (0,2,4,5,6,8,9,10,12,13,14,16,17,18,20,21,22,24,25,26,28,29,30,32,33,34,36,37,38,40,41,42,44,45,46,48,49,50,52,53,54,56,57,58,60,61,62,64,65,66,68,69,70,72,73,74,76,77,78,80,81,82,84,85,86,88,89,90,92,93,94,96,97,98,100,101,102,104,105,106,108,109,110,112,113,114,116,117,118,120,121,122,124,125,126,128,129) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('data.csv', sep='\t')


Dataset has been successfully loaded.
  Unnamed: 0        date     id size_grp       age   aliq_at aliq_mat  \
0      53170   9/30/1962  33814     mega -0.410891  0.343616      0.0   
1      53907  10/31/1962  33814     mega  -0.41374  0.333333      0.0   
2      54659  11/30/1962  33814     mega -0.401786  0.331697      0.0   
3      55425  12/31/1962  33814     mega  -0.37069  0.322252      0.0   
4      56207   1/31/1963  33814     mega -0.367951  0.324561      0.0   

   ami_126d     at_be    at_gr1  ... turnover_var_126d  z_score  \
0 -0.290922 -0.447445  0.247967  ...          0.274763      0.0   
1 -0.281893 -0.447598  0.231343  ...          0.210598      0.0   
2 -0.297587 -0.448791  0.221485  ...          0.167995      0.0   
3 -0.309585 -0.439808  0.215019  ...           0.12516      0.0   
4 -0.308729 -0.441892  0.210828  ...          0.035714      0.0   

  zero_trades_126d zero_trades_21d zero_trades_252d       ret Unnamed: 126  \
0         0.178426        0.033875        

Part 2 :Sanity Checks

In [31]:
# Check for constant columns
constant_cols = numeric_cols[data[numeric_cols].std() == 0]
print("Constant columns:")
print(constant_cols)

# Check for duplicate rows
duplicate_rows = data.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows}")


# Check for non-numeric values in numeric columns
for col in numeric_cols:
    if data[col].isnull().any():
        print(f"Column {col} contains null values.")
    else:
        print(f"Column {col} does not contain null values.")

# Check date format consistency
date_col = 'date'  # Replace 'date' with the actual column name
try:
    pd.to_datetime(data[date_col])
except ValueError:
    print("Date formats are not consistent.")
else:
    print("Date formats appear to be consistent.")

# Check for extreme values (highly skewed distributions)

for col in numeric_cols:
    skewness = stats.skew(data[col])
    if abs(skewness) > 2:
        print(f"Column {col} has a highly skewed distribution (skewness: {skewness}).")
    else:
        print(f"Column {col} does not have a highly skewed distribution (skewness: {skewness}).")

Constant columns:
Index(['Unnamed: 127'], dtype='object')
Duplicate rows: 0
Column ami_126d contains null values.
Column at_turnover contains null values.
Column beta_dimson_21d contains null values.
Column bidaskhl_21d contains null values.
Column coa_gr1a contains null values.
Column corr_1260d contains null values.
Column debt_me contains null values.
Column dolvol_var_126d contains null values.
Column ebitda_mev contains null values.
Column eqnpo_12m contains null values.
Column fnl_gr1a contains null values.
Column iskew_capm_21d contains null values.
Column ivol_capm_252d contains null values.
Column lnoa_gr1a contains null values.
Column mispricing_perf contains null values.
Column netis_at contains null values.
Column niq_at contains null values.
Column noa_at contains null values.
Column oaccruals_ni contains null values.
Column op_at contains null values.
Column ppeinv_gr1a contains null values.
Column qmj_safety contains null values.
Column ret_12_1 contains null values.
Col

Part 3 : Clean Dataset

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Create a copy of the original data
cleanedData = data.copy()

print("Original shape:", cleanedData.shape)

# Define numeric columns if not already defined
numeric_cols = [col for col in data.columns if data[col].dtype.kind in 'bifc']

# Remove constant columns
constant_cols = [col for col in cleanedData.columns if cleanedData[col].nunique() == 1]
cleanedData = cleanedData.drop(constant_cols, axis=1)
print(f"After removing constant columns shape: {cleanedData.shape}")

# Update numeric_cols to exclude removed columns
numeric_cols = [col for col in numeric_cols if col in cleanedData.columns]

# Print a sample of the date column to understand its format
print("\nSample of date column:")
print(cleanedData['date'].head())
print("Date column type:", cleanedData['date'].dtype)

# SAFER DATE HANDLING - don't drop rows, just convert what we can
try:
    # Try simple conversion first
    cleanedData['date_converted'] = pd.to_datetime(cleanedData['date'], errors='coerce')
    
    # Check how many dates were successfully converted
    conversion_success = (~pd.isna(cleanedData['date_converted'])).sum()
    conversion_rate = conversion_success / len(cleanedData) * 100
    
    print(f"Successfully converted {conversion_success} dates ({conversion_rate:.2f}%)")
    
    # If conversion rate is good, use the converted dates
    if conversion_rate > 90:
        cleanedData['date'] = cleanedData['date_converted']
        cleanedData = cleanedData.drop('date_converted', axis=1)
        print("Used converted dates")
    else:
        # Keep original dates and the converted ones for reference
        print("Keeping both original and converted dates")
    
    date_col = 'date'  # Use the original date column name
    
except Exception as e:
    print(f"Error in date conversion: {e}")
    date_col = 'date'  # Keep using the original column

print(f"After date handling shape: {cleanedData.shape}")

# Handle non-numeric values in numeric columns
for col in numeric_cols:
    if cleanedData[col].isnull().any():
        # Fill null values with the mean of the column
        cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mean())
    else:
        print(f"Column {col} does not contain null values.")

# Check for NaN values in the dataset
nan_cols = cleanedData.columns[cleanedData.isnull().any()].tolist()
print("Columns with NaN values:", nan_cols)

# Fill NaN values with the mean of the column (for numerical columns)
for col in nan_cols:
    if cleanedData[col].dtype.kind in 'bifc':  # Check if column is numeric
        if not cleanedData[col].empty:
            cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mean())
        else:
            print(f"Column {col} is empty and cannot be filled with the mean.")
    else:
        # Fill NaN values with the mode of the column (for categorical columns)
        if not cleanedData[col].empty:
            cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mode().iloc[0])
        else:
            print(f"Column {col} is empty and cannot be filled with the mode.")

print(f"After filling NaN values shape: {cleanedData.shape}")

# Check if there are still NaN values in the dataset
if cleanedData.isnull().values.any():
    print("There are still NaN values in the dataset.")
    nan_cols = cleanedData.columns[cleanedData.isnull().any()].tolist()
    print("Columns with NaN values:", nan_cols)
else:
    print("All NaN values have been filled.")

# Handle extreme values (highly skewed distributions) - SAFER VERSION
for col in numeric_cols:
    try:
        # Check if the column has any negative values before applying log transformation
        if cleanedData[col].min() < 0:
            # Use winsorization for skewed data with negative values
            q_low = cleanedData[col].quantile(0.01)
            q_high = cleanedData[col].quantile(0.99)
            cleanedData[col] = cleanedData[col].clip(q_low, q_high)
            print(f"Applied winsorization to {col}")
        else:
            # Only calculate skewness if we have enough data
            if cleanedData[col].count() > 8:  # Minimum sample size
                skewness = stats.skew(cleanedData[col].dropna())
                if abs(skewness) > 2:
                    # Apply logarithmic transformation to reduce skewness
                    cleanedData[col] = np.log1p(cleanedData[col])
                    print(f"Applied log transformation to {col} (skewness: {skewness:.2f})")
    except Exception as e:
        print(f"Error processing column {col}: {e}")

print(f"Final cleaned shape: {cleanedData.shape}")

# Check the first few rows to understand the data
print("\nFirst 5 rows of cleaned data:")
print(cleanedData.head())

# Check data types
print("\nCleaned data types:")
print(cleanedData.dtypes)


#Additional cleanup, maybe not needed :

# Save the cleaned data if needed
cleanedData.to_csv('cleaned_financial_data.csv')
# print("Saved cleaned data to file")



Original shape: (122984, 130)
After removing constant columns shape: (122984, 128)

Sample of date column:
0     9/30/1962
1    10/31/1962
2    11/30/1962
3    12/31/1962
4     1/31/1963
Name: date, dtype: object
Date column type: object
Successfully converted 122978 dates (100.00%)
Used converted dates
After date handling shape: (122984, 128)
Columns with NaN values: ['Unnamed: 0', 'date', 'id', 'size_grp', 'age', 'aliq_at', 'aliq_mat', 'at_be', 'at_gr1', 'at_me', 'be_gr1a', 'be_me', 'beta_60m', 'betabab_1260d', 'betadown_252d', 'bev_mev', 'capx_gr1', 'cash_at', 'chcsho_12m', 'col_gr1a', 'cop_at', 'cop_atl1', 'coskew_21d', 'cowc_gr1a', 'dbnetis_at', 'dgp_dsale', 'div12m_me', 'dolvol_126d', 'dsale_drec', 'ebit_bev', 'ebit_sale', 'emp_gr1', 'eq_dur', 'eqnetis_at', 'eqnpo_me', 'eqpo_me', 'fcf_me', 'gp_at', 'gp_atl1', 'inv_gr1a', 'iskew_ff3_21d', 'iskew_hxz4_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'ivol_hxz4_21d', 'kz_index', 'lti_gr1a', 'market_equity', 'mispricing_mgmt', 'ncoa_gr1a', 'nc

/tmp/ipykernel_3546/3469560960.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cleanedData[col + "_processed"] = True
/tmp/ipykernel_3546/3469560960.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cleanedData[col + "_processed"] = True
/tmp/ipykernel_3546/3469560960.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, 

Unnamed columns: ['Unnamed: 0', 'Unnamed: 126', 'Unnamed: 128']
Consider removing these columns if they're not needed for your analysis.
Set multi-index of date and id
Sorted index
Final cleaned shape: (122984, 126)

Final data types:
Unnamed: 0          float64
size_grp             object
age                 float64
aliq_at             float64
aliq_mat            float64
                     ...   
zero_trades_21d     float64
zero_trades_252d    float64
ret                 float64
Unnamed: 126         object
Unnamed: 128        float64
Length: 126, dtype: object

Preview of cleaned data:
                  Unnamed: 0 size_grp       age   aliq_at  aliq_mat  ami_126d  \
date       id                                                                   
1962-09-30 33814   153443.47     mega -0.410891  0.343616       0.0 -0.290922   
1962-10-31 33814   153443.47     mega -0.413740  0.333333       0.0 -0.281893   
1962-11-30 33814   153443.47     mega -0.401786  0.331697       0.0 -0.297587   

Part 3 : Linear Regression

In [22]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from scipy import stats

# First, let's convert the data to numeric types where needed
# Create a clean copy of the data
cleanedData = data.copy()

# Convert columns to numeric where possible
for col in cleanedData.columns:
    if col not in ['date', 'id', 'size_grp']:  # Skip non-numeric columns
        try:
            cleanedData[col] = pd.to_numeric(cleanedData[col])
        except:
            print(f"Could not convert {col} to numeric")

# Identify the Book-to-Market and returns columns
btm_col = 'be_me'  # Book-to-Market column (index 13)
returns_col = 'ret'  # Returns column (index 125)

# Remove extreme outliers (beyond 3 standard deviations)
for col in [btm_col, returns_col]:
    mean = cleanedData[col].mean()
    std = cleanedData[col].std()
    cleanedData = cleanedData[(cleanedData[col] > mean - 3*std) & (cleanedData[col] < mean + 3*std)]

print(f"After cleaning, dataset shape: {cleanedData.shape}")

# 1. Scatter Plot of Book-to-Market and Returns
plt.figure(figsize=(10, 6))
plt.scatter(cleanedData[btm_col], cleanedData[returns_col], alpha=0.5)
plt.title('Book-to-Market vs Returns')
plt.xlabel('Book-to-Market Ratio')
plt.ylabel('Returns')
plt.grid(True, linestyle='--', alpha=0.7)

# Add a trend line
z = np.polyfit(cleanedData[btm_col], cleanedData[returns_col], 1)
p = np.poly1d(z)
plt.plot(cleanedData[btm_col], p(cleanedData[btm_col]), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

print(f"Correlation between Book-to-Market and Returns: {cleanedData[btm_col].corr(cleanedData[returns_col]):.4f}")

# 2. Linear Regression with sklearn
# Prepare data for regression
X = cleanedData[btm_col].values.reshape(-1, 1)
y = cleanedData[returns_col].values

# Fit the model
model = LinearRegression()
model.fit(X, y)

# Get predictions
y_pred = model.predict(X)

# Print results
print("\nLinear Regression Results (sklearn):")
print(f"Coefficient (slope): {model.coef_[0]:.6f}")
print(f"Intercept: {model.intercept_:.6f}")
print(f"R-squared: {r2_score(y, y_pred):.6f}")
print(f"Mean Squared Error: {mean_squared_error(y, y_pred):.6f}")

# Use statsmodels for p-values
X_sm = sm.add_constant(X)  # Add a constant for the intercept
model_sm = sm.OLS(y, X_sm).fit()

# Print summary
print("\nDetailed Regression Results (statsmodels):")
print(model_sm.summary())

# 3. Verify Results with Manual OLS Calculation
# Manual OLS calculation
# Formula: β = (X'X)^(-1)X'y

# Prepare data
X_manual = sm.add_constant(cleanedData[btm_col])  # Add constant
y_manual = cleanedData[returns_col]

# Calculate β using the OLS formula
X_transpose_X = X_manual.T @ X_manual
X_transpose_y = X_manual.T @ y_manual
beta = np.linalg.inv(X_transpose_X) @ X_transpose_y

# Calculate predictions
y_pred_manual = X_manual @ beta

# Calculate R-squared
y_mean = np.mean(y_manual)
ss_total = np.sum((y_manual - y_mean) ** 2)
ss_residual = np.sum((y_manual - y_pred_manual) ** 2)
r_squared_manual = 1 - (ss_residual / ss_total)

# Calculate standard errors
n = len(y_manual)
k = X_manual.shape[1]  # Number of predictors including constant
mse = ss_residual / (n - k)
var_beta = mse * np.linalg.inv(X_transpose_X)
se_beta = np.sqrt(np.diag(var_beta))

# Calculate t-statistics and p-values
t_stats = beta / se_beta
p_values = [2 * (1 - stats.t.cdf(abs(t), n - k)) for t in t_stats]

print("\nManual OLS Results:")
print(f"Coefficients: Intercept = {beta[0]:.6f}, Slope = {beta[1]:.6f}")
print(f"Standard Errors: Intercept = {se_beta[0]:.6f}, Slope = {se_beta[1]:.6f}")
print(f"t-statistics: Intercept = {t_stats[0]:.4f}, Slope = {t_stats[1]:.4f}")
print(f"p-values: Intercept = {p_values[0]:.6f}, Slope = {p_values[1]:.6f}")
print(f"R-squared: {r_squared_manual:.6f}")

# Compare with sklearn results
print("\nComparison with sklearn:")
print(f"Coefficient difference: {abs(model.coef_[0] - beta[1]):.10f}")
print(f"Intercept difference: {abs(model.intercept_ - beta[0]):.10f}")
print(f"R-squared difference: {abs(r2_score(y, y_pred) - r_squared_manual):.10f}")

# 4. Can We Make Money with This Strategy?
# Analyze the potential profitability
print("\nProfitability Analysis:")

# Calculate average returns by Book-to-Market quintiles
cleanedData['btm_quintile'] = pd.qcut(cleanedData[btm_col], 5, labels=False)
quintile_returns = cleanedData.groupby('btm_quintile')[returns_col].mean()

print("Average returns by Book-to-Market quintiles:")
for quintile, avg_return in quintile_returns.items():
    print(f"Quintile {quintile+1}: {avg_return:.6f}")

# Calculate the spread between highest and lowest quintiles
spread = quintile_returns.iloc[-1] - quintile_returns.iloc[0]
print(f"\nSpread (Q5-Q1): {spread:.6f}")

# Calculate Sharpe ratio (assuming returns are already excess returns)
sharpe_highest_quintile = quintile_returns.iloc[-1] / cleanedData[cleanedData['btm_quintile'] == 4][returns_col].std()
print(f"Sharpe ratio of highest quintile: {sharpe_highest_quintile:.4f}")

# Discuss limitations
print("\nLimitations of this strategy:")
print("1. Transaction costs not accounted for")
print("2. Look-ahead bias may be present")
print("3. Market frictions (liquidity, short-selling constraints)")
print("4. Time-varying relationship between Book-to-Market and returns")
print("5. Other risk factors not controlled for (size, momentum, etc.)")
print("6. The low R-squared indicates that Book-to-Market explains only a small portion of return variation")

# 5. Market Timing Strategy with Five Stocks
print("\nMarket Timing Strategy with Five Stocks:")

# Check if we have stock identifiers
if 'id' in cleanedData.columns:
    # We have stock identifiers
    print("Using 'id' as stock identifier")
    
    # Get the most frequent stocks
    stock_counts = cleanedData['id'].value_counts()
    eligible_stocks = stock_counts[stock_counts > 20].index.tolist()  # Stocks with at least 20 observations
    
    if len(eligible_stocks) >= 5:
        # Select 5 random stocks
        np.random.seed(42)  # For reproducibility
        random_stocks = np.random.choice(eligible_stocks, 5, replace=False)
        print(f"Selected stocks: {random_stocks}")
        
        # Filter data for selected stocks
        stock_data = cleanedData[cleanedData['id'].isin(random_stocks)]
        
        # Create a pivot table with returns for each stock
        stock_returns = stock_data.pivot_table(index='date', columns='id', values=returns_col)
        
        # Create a pivot table with Book-to-Market for each stock
        stock_btm = stock_data.pivot_table(index='date', columns='id', values=btm_col)
        
        # Create a DataFrame for our strategy
        strategy_data = pd.DataFrame(index=stock_returns.index)
        
        # Calculate market return (equal-weighted average of all stocks)
        market_returns = stock_returns.mean(axis=1)
        strategy_data['market_ret'] = market_returns
        
        # For each stock, predict returns based on its Book-to-Market
        for stock in stock_returns.columns:
            strategy_data[f'btm_{stock}'] = stock_btm[stock]
            strategy_data[f'actual_ret_{stock}'] = stock_returns[stock]
            strategy_data[f'pred_ret_{stock}'] = model.intercept_ + model.coef_[0] * stock_btm[stock]
        
        # Create signals (1 if predicted return > median, 0 otherwise)
        for stock in stock_returns.columns:
            strategy_data[f'signal_{stock}'] = (strategy_data[f'pred_ret_{stock}'] > strategy_data[f'pred_ret_{stock}'].median()).astype(int)
        
        # Calculate strategy returns (invest in stocks when signal is 1, otherwise in risk-free asset)
        # Assuming risk-free rate is 0 for simplicity
        for stock in stock_returns.columns:
            strategy_data[f'strategy_{stock}'] = strategy_data[f'signal_{stock}'] * strategy_data[f'actual_ret_{stock}']
        
        # Calculate portfolio return (equal-weighted)
        strategy_data['portfolio_ret'] = strategy_data[[f'strategy_{stock}' for stock in stock_returns.columns]].mean(axis=1)
        
        # Calculate cumulative returns
        strategy_data['cum_portfolio_ret'] = (1 + strategy_data['portfolio_ret']).cumprod() - 1
        strategy_data['cum_market_ret'] = (1 + strategy_data['market_ret']).cumprod() - 1
        
        # Plot cumulative returns
        plt.figure(figsize=(12, 6))
        plt.plot(strategy_data['cum_portfolio_ret'], label='Strategy')
        plt.plot(strategy_data['cum_market_ret'], label='Market')
        plt.title('Cumulative Returns: Strategy vs Market')
        plt.xlabel('Time')
        plt.ylabel('Cumulative Return')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
        
        # Calculate performance metrics
        strategy_return = strategy_data['portfolio_ret'].mean()
        strategy_volatility = strategy_data['portfolio_ret'].std()
        strategy_sharpe = strategy_return / strategy_volatility
        
        market_return = strategy_data['market_ret'].mean()
        market_volatility = strategy_data['market_ret'].std()
        market_sharpe = market_return / market_volatility
        
        print("\nPerformance Metrics:")
        print(f"Strategy - Return: {strategy_return:.6f}, Volatility: {strategy_volatility:.6f}, Sharpe: {strategy_sharpe:.4f}")
        print(f"Market - Return: {market_return:.6f}, Volatility: {market_volatility:.6f}, Sharpe: {market_sharpe:.4f}")
        
        # Calculate alpha and beta
        X_market = sm.add_constant(strategy_data['market_ret'])
        model_alpha_beta = sm.OLS(strategy_data['portfolio_ret'], X_market).fit()
        alpha = model_alpha_beta.params[0]
        beta = model_alpha_beta.params[1]
        
        print(f"\nAlpha: {alpha:.6f} (p-value: {model_alpha_beta.pvalues[0]:.4f})")
        print(f"Beta: {beta:.6f} (p-value: {model_alpha_beta.pvalues[1]:.4f})")
        
        # Conclusion
        print("\nConclusion:")
        if strategy_sharpe > market_sharpe:
            print("The strategy outperforms the market in terms of risk-adjusted returns.")
            if alpha > 0 and model_alpha_beta.pvalues[0] < 0.05:
                print("The strategy generates statistically significant alpha.")
            else:
                print("However, the alpha is not statistically significant.")
        else:
            print("The strategy does not outperform the market in terms of risk-adjusted returns.")
            
        print("\nLimitations of the backtest:")
        print("1. In-sample testing (no out-of-sample validation)")
        print("2. Transaction costs not accounted for")
        print("3. Limited number of stocks in the portfolio")
        print("4. Simplistic signal generation")
        print("5. Look-ahead bias may be present")
    else:
        print(f"Not enough eligible stocks. Found only {len(eligible_stocks)} stocks with sufficient data.")
else:
    print("Stock identifier column not found. Cannot implement stock-specific strategy.")


Could not convert Unnamed: 0 to numeric
Could not convert age to numeric
Could not convert aliq_at to numeric
Could not convert aliq_mat to numeric
Could not convert at_be to numeric
Could not convert at_gr1 to numeric
Could not convert at_me to numeric
Could not convert be_gr1a to numeric
Could not convert be_me to numeric
Could not convert beta_60m to numeric
Could not convert betabab_1260d to numeric
Could not convert betadown_252d to numeric
Could not convert bev_mev to numeric
Could not convert capx_gr1 to numeric
Could not convert cash_at to numeric
Could not convert chcsho_12m to numeric
Could not convert col_gr1a to numeric
Could not convert cop_at to numeric
Could not convert cop_atl1 to numeric
Could not convert coskew_21d to numeric
Could not convert cowc_gr1a to numeric
Could not convert dbnetis_at to numeric
Could not convert dgp_dsale to numeric
Could not convert div12m_me to numeric
Could not convert dolvol_126d to numeric
Could not convert dsale_drec to numeric
Could no

TypeError: unsupported operand type(s) for +: 'float' and 'str'